# <font color='blue'> Random forest classification model </font>

####  <font color='blue'> Sabah Sabaghy 15 July 2026

"""

Satellite Image Classification using Random Forest (Parallel Optimised Version)

This workflow classifies multi-band satellite image tiles into land cover classes using a Random Forest model trained from ground truth data. The script reads satellite imagery, training data, and masks, predicts land cover classes, and exports classified maps and class probability layers.

The workflow has been further developed and optimised to improve performance through faster multi-band image reading and parallel processing of image tiles. These enhancements significantly reduce processing time for large datasets while preserving consistency with the original classification workflow.

Note: Satellite, training, and mask data tiles must share the same filename and tile ID for correct processing.

"""

## Read input data, including ground truth data and images

In [ ]:
from osgeo import gdal, gdal_array
import math
import os
import glob
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
%matplotlib inline
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
import dask_ml.model_selection as dcv
from dask.distributed import Client, LocalCluster
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,  cohen_kappa_score)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Tell GDAL to throw Python exceptions, and register all drivers
gdal.UseExceptions()
gdal.AllRegister()

In [ ]:
%%time

# =============================================================================
# Input folders
# =============================================================================

dir_sat = r"..."
dir_train = r"..."

# =============================================================================
# Find files
# =============================================================================

sat_files = sorted(glob.glob(os.path.join(dir_sat, "*.tif")))
train_files = sorted(glob.glob(os.path.join(dir_train, "*.tif")))

print(f"Number of tiles to process: {len(sat_files)}")

if len(sat_files) != len(train_files):
    raise ValueError(
        f"Different number of satellite ({len(sat_files)}) "
        f"and training ({len(train_files)}) tiles."
    )

# =============================================================================
# Check first tile
# =============================================================================

with rasterio.open(sat_files[0]) as src:
    arr = src.read(1)

print("\nFirst tile diagnostics")
print("----------------------")
print("Bands      :", src.count)
print("Shape      :", arr.shape)
print("Data type  :", arr.dtype)
print("Min value  :", arr.min())
print("Max value  :", arr.max())

# =============================================================================
# Extract training pixels
# =============================================================================

X_list = []
y_list = []

for i, (sat_file, train_file) in enumerate(
    zip(sat_files, train_files), start=1
):

    print(
        f"Processing tile {i}/{len(sat_files)} : "
        f"{os.path.basename(sat_file)}"
    )

    # -------------------------------------------------------------------------
    # Read satellite image
    # -------------------------------------------------------------------------

    with rasterio.open(sat_file) as sat:

        # Keep as int16 to reduce memory
        img = sat.read()

    # (bands, rows, cols) -> (pixels, bands)
    img = img.reshape(img.shape[0], -1).T

    # -------------------------------------------------------------------------
    # Read training raster
    # -------------------------------------------------------------------------

    with rasterio.open(train_file) as roi:

        labels = roi.read(1).ravel()

        nodata = roi.nodatavals[0]

    # -------------------------------------------------------------------------
    # Remove NoData pixels
    # -------------------------------------------------------------------------

    if nodata is not None:
        mask = labels != nodata
    else:
        mask = ~np.isnan(labels)

    X_tile = img[mask]
    y_tile = labels[mask]

    print(
        f"   Valid training samples: "
        f"{len(y_tile):,}"
    )

    X_list.append(X_tile)
    y_list.append(y_tile)

# =============================================================================
# Combine all tiles
# =============================================================================

print("\nCombining tiles...")

X = np.vstack(X_list)
y = np.concatenate(y_list)

print(f"Total training samples: {len(y):,}")
print(f"Number of bands: {X.shape[1]}")

# =============================================================================
# Create DataFrame
# =============================================================================

band_names = [f"band{i}" for i in range(1, X.shape[1] + 1)]

df = pd.DataFrame(X, columns=band_names)

df["Training_Data"] = y

print("\nFinal dataframe shape:")
print(df.shape)

display(df.head())

In [ ]:
# remove duplicate rows based on all columns
df = df.drop_duplicates()
df.to_excel(r"...\test.xlsx") 
df

In [ ]:
df['Training_Data'].unique()

In [ ]:
df = pd.read_excel(r"...\test.xlsx")
# remove frist column which referes to index 
df = df.iloc[:, 1:]
df

# Concatenate training datasets

## Using Skicit-learn to split ground truth and image data into training and testing sets

In [ ]:
%%time

features = df.drop(columns=['Training_Data'], axis=1) 
labels = df['Training_Data']

# Split the data into training and testing sets
# train_test_split: Allowed inputs are lists, numpy arrays, scipy-sparse matrices or pandas dataframes.
train_features, test_features, train_labels, test_labels = train_test_split(features, labels, test_size = 0.30, random_state = 42)

## Make sure the split of data is correct

In [ ]:
print('Training Features Shape:', train_features.shape)
print('Training Labels Shape:', train_labels.shape)
print('Testing Features Shape:', test_features.shape)
print('Testing Labels Shape:', test_labels.shape)

## Model Tuning

In [ ]:
# =============================================================================
# select hyperparameters (hyperparameters are sorted based on their importance; number od estimators/trees is the most important parameter)
# =============================================================================

# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 5)]
# Number of features to consider at every split
max_features = ['log2', 'sqrt']
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(5, 25, num = 5)]
# Minimum number of samples required to split a node
min_samples_split = [2,5,10,15]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4, 10]
bootstrap = [True, False]
# oob_score = [True, False]

hyperF = dict(n_estimators = n_estimators, max_depth = max_depth,  
              min_samples_split = min_samples_split, min_samples_leaf = min_samples_leaf,
             max_features = max_features, bootstrap = bootstrap) #, oob_score = oob_score

### Finding best machine learning model hyperparameters
#### <font color='blue'>**Randomized Search CV:** </font>
Random Search sets up a grid of hyperparameter values and selects random combinations to train the model and score. This allows you to explicitly control the number of parameter combinations that are attempted. The number of search iterations is set based on time or resources.<br>
<font color='green'> **Pros**: reduced overfitting problem, more accurate long term results </font>

In [ ]:
%%time
# =============================================================================
# Start Dask cluster
# =============================================================================

cluster = LocalCluster(
    n_workers=24, #18 random forest performs better with fewer, larger workers.
    threads_per_worker=2, #4
    processes=True
)

client = Client(cluster)

print(client)

# =============================================================================
# Initialize a Random Forest Model
# =============================================================================

rf = RandomForestClassifier(n_jobs=1, random_state=42)

randomF = dcv.RandomizedSearchCV(
    estimator=rf,
    param_distributions=hyperF,
    n_iter=20,
    cv=3,
    random_state=42
)

# =============================================================================
# Fit model
# =============================================================================

bestF = randomF.fit(train_features, train_labels)
print("The mean accuracy of the model is:", bestF.score(test_features, test_labels))

In [ ]:
# Save the trained random forest model
joblib.dump(bestF, "./test.joblib")

In [ ]:
# get the best estimator that was chosen by the search
bestF.best_estimator_

In [ ]:
# check the cv results
import pandas as pd
pd.DataFrame(bestF.cv_results_)

## Evaluate model performance

In [ ]:
def evaluate(model, test_features, test_labels):
    
    # Predict
    predictions = model.predict(test_features)

    # Metrics
    oa = accuracy_score(test_labels, predictions)
    kappa = cohen_kappa_score(test_labels, predictions)

    
    print("Confusion Matrix:")
    print(confusion_matrix(test_labels, predictions))

    print("\nClassification Report:")
    print(classification_report(test_labels, predictions))

    print(f"\nOverall Accuracy: {oa * 100:.2f}%")
    print(f"Kappa: {kappa:.4f}")
    
    return predictions

evalu = evaluate(bestF, test_features, test_labels)

## Visualize confusion matrix 

In [ ]:

labels_1 = {'3':'Deciduous fruit tree','4': 'Evergreen fruit tree', '5': 'Hardwood plantation', '6': 'Softwood plantation',
             '7': 'Pastures and grassland', '8': 'Cereals', '9': 'Legumes', '10': 'Oilseeds',
             '11': 'Vegetable and herbs', '12': 'Native trees and shrubland'} 



label_temp = list(range(3,13))


labels_list = [labels_1[key] for key in labels_1]

cm = confusion_matrix(test_labels, evalu, labels=label_temp)
# print(cm)

cmd = ConfusionMatrixDisplay(cm, display_labels=labels_1.values())
cmd.plot(xticks_rotation='vertical', cmap='Greys')
plt.show()

## Find the importance of each satellite image band

In [ ]:
bands = list(range(31))

for b, imp in zip(bands, bestF.best_estimator_.feature_importances_):
    print('Band {b} importance: {imp}'.format(b = b, imp = round(imp, 2)))

## Classify tiles from a satellite image based on a pre-trained Random Forest model

In [ ]:
%%time
# Assign directory where input imagery files (tiles) to be classified are located
dir_img = r"..."
dir_mask = r"..."

# Assign directories to save output rasters
dir_class = r"..."

# Assign directories to save output rasters
dir_prob = r"..."

# Count number of files to classify
count = 0
for path in os.listdir(dir_img):
    if os.path.isfile(os.path.join(dir_img, path)):
        count += 1
print("Images to process = " + str(count))

# Import saved Random Forest model
loaded_rf = joblib.load(r"...\test.joblib")

In [ ]:
%%time

# =============================================================================
# Setting
# =============================================================================

# Enter model name to be used in output file names
modelname = "model"

os.makedirs(dir_class, exist_ok=True)


# Number of tiles to process in parallel
# Start with 2 or 4. Increase only if RAM and disk I/O are fine.
tile_workers = 2

# Notes: if tile_workers = 4, running the classification takes about 1 day and 21 hours
# if tile_workers = 6, running the classification takes about 2 days and 1 hours
# if tile_workers = 2, running the classification takes about about 1 day and 20 hours

# Output NoData value
out_nodata = -9999

# Output data type
out_dtype = "int16"

# =============================================================================
# Avoid nested parallelism
# =============================================================================
# If we process several tiles in parallel, each Random Forest prediction should
# not also try to use all CPU cores.

try:
    loaded_rf.set_params(n_jobs=1)
except Exception:
    pass

# =============================================================================
# Find image and mask files
# =============================================================================

img_files = sorted(glob.glob(os.path.join(dir_img, "*.tif")))

mask_files = {
    os.path.basename(p): p
    for p in glob.glob(os.path.join(dir_mask, "**", "*.tif"), recursive=True)
}

print(f"Number of image tiles found: {len(img_files)}")
print(f"Number of mask tiles found : {len(mask_files)}")
print(f"Using {tile_workers} parallel tile workers")

# =============================================================================
# Check expected number of bands
# =============================================================================

with rasterio.open(img_files[0]) as src:
    expected_bands = src.count
    print("\nFirst image diagnostics")
    print("-----------------------")
    print("File       :", os.path.basename(img_files[0]))
    print("Bands      :", src.count)
    print("Shape      :", src.height, src.width)
    print("Data types :", src.dtypes)

# If model stores expected feature count, check it
if hasattr(loaded_rf, "n_features_in_"):
    if loaded_rf.n_features_in_ != expected_bands:
        raise ValueError(
            f"Model expects {loaded_rf.n_features_in_} features, "
            f"but image has {expected_bands} bands."
        )

# =============================================================================
# Function to classify one tile
# =============================================================================

def classify_tile(img_file):

    img_base = os.path.basename(img_file)

    # Your original naming logic:
    # filename[-8:] gives something like "7021.tif"
    # then adding "t" gives "t7021.tif"
    mask_name = "t" + img_base[-8:]

    if mask_name not in mask_files:
        return f"SKIPPED: mask not found for {img_base} -> expected {mask_name}"

    mask_file = mask_files[mask_name]

    # -------------------------------------------------------------------------
    # Read image
    # -------------------------------------------------------------------------

    with rasterio.open(img_file) as img:

        if img.count != expected_bands:
            raise ValueError(
                f"{img_base} has {img.count} bands, "
                f"expected {expected_bands}."
            )

        # Read all bands at once
        # Shape: (bands, rows, cols)
        arr = img.read()

        rows = img.height
        cols = img.width

        # Prepare output profile
        profile = img.profile.copy()
        profile.update(
            count=1,
            dtype=out_dtype,
            nodata=out_nodata,
            compress="lzw"
        )

    # Convert:
    # (bands, rows, cols) -> (pixels, bands)
    X_all = arr.reshape(arr.shape[0], -1).T

    # -------------------------------------------------------------------------
    # Read mask
    # -------------------------------------------------------------------------

    with rasterio.open(mask_file) as msk:

        mask_arr = msk.read(1).ravel()
        mask_nodata = msk.nodatavals[0]

    
    if mask_arr.shape[0] != X_all.shape[0]:
        raise ValueError(
            f"Image and mask size mismatch for {img_base}: "
            f"image {X_all.shape[0]}, "
            f"mask pixels = {mask_arr.shape[0]}"
        )

    # -------------------------------------------------------------------------
    # Valid prediction area
    # -------------------------------------------------------------------------

    if mask_nodata is not None:
        valid = mask_arr != mask_nodata
    else:
        valid = ~np.isnan(mask_arr)

    n_valid = int(valid.sum())

    # Create output filled with NoData
    out = np.full(mask_arr.shape, out_nodata, dtype=out_dtype)

    # -------------------------------------------------------------------------
    # Predict only valid pixels
    # -------------------------------------------------------------------------

    if n_valid > 0:

        X_valid = X_all[valid]

        predictions = loaded_rf.predict(X_valid)

        out[valid] = predictions.astype(out_dtype)

    # Reshape back to raster
    class_pred = out.reshape(rows, cols)

    # -------------------------------------------------------------------------
    # Save output
    # -------------------------------------------------------------------------

    imgname = os.path.splitext(img_base)[0]

    out_path = os.path.join(
        dir_class,
        f"{imgname}_{modelname}.tif"
    )

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(class_pred, 1)

    return f"DONE: {out_path} | valid pixels: {n_valid:,}"

# =============================================================================
# Run classification in parallel
# =============================================================================

results = []

with ThreadPoolExecutor(max_workers=tile_workers) as executor:

    futures = {
        executor.submit(classify_tile, img_file): img_file
        for img_file in img_files
    }

    for i, future in enumerate(as_completed(futures), start=1):

        try:
            message = future.result()
            results.append(message)
            print(f"{i}/{len(img_files)} - {message}")

        except Exception as e:
            img_file = futures[future]
            print(f"{i}/{len(img_files)} - FAILED: {os.path.basename(img_file)}")
            print(e)

print("\nClassification completed.")


## Calculate probaility of each class

In [ ]:
%%time

# =============================================================================
# Setting
# =============================================================================

modelname = "model"

os.makedirs(dir_prob, exist_ok=True)

tile_workers = 2

out_nodata = -9999.0
out_dtype = "float32"

# =============================================================================
# Avoid nested parallelism
# =============================================================================

try:
    loaded_rf.set_params(n_jobs=1)
except Exception:
    pass

# =============================================================================
# Find image and mask files
# =============================================================================

img_files = sorted(glob.glob(os.path.join(dir_img, "*.tif")))

mask_files = {
    os.path.basename(p): p
    for p in glob.glob(os.path.join(dir_mask, "**", "*.tif"), recursive=True)
}

print(f"Number of image tiles found: {len(img_files)}")
print(f"Number of mask tiles found : {len(mask_files)}")
print(f"Using {tile_workers} parallel tile workers")

# =============================================================================
# Check expected number of bands
# =============================================================================

with rasterio.open(img_files[0]) as src:

    expected_bands = src.count

    print("\nFirst image diagnostics")
    print("-----------------------")
    print("File       :", os.path.basename(img_files[0]))
    print("Bands      :", src.count)
    print("Shape      :", src.height, src.width)
    print("Data types :", src.dtypes)

if hasattr(loaded_rf, "n_features_in_"):

    if loaded_rf.n_features_in_ != expected_bands:

        raise ValueError(
            f"Model expects {loaded_rf.n_features_in_} features, "
            f"but image has {expected_bands} bands."
        )

# =============================================================================
# Function to calculate probability for one tile
# =============================================================================

def probability_tile(img_file):

    img_base = os.path.basename(img_file)

    mask_name = "t" + img_base[-8:]

    if mask_name not in mask_files:
        return f"SKIPPED: mask not found for {img_base} -> expected {mask_name}"

    mask_file = mask_files[mask_name]

    # -------------------------------------------------------------------------
    # Read image
    # -------------------------------------------------------------------------

    with rasterio.open(img_file) as img:

        if img.count != expected_bands:

            raise ValueError(
                f"{img_base} has {img.count} bands, "
                f"expected {expected_bands}."
            )

        arr = img.read()

        rows = img.height
        cols = img.width

        profile = img.profile.copy()

        profile.update(
            count=1,
            dtype=out_dtype,
            nodata=out_nodata,
            compress="lzw"
        )

    # Convert from:
    # (bands, rows, cols)
    # to:
    # (pixels, bands)

    X_all = arr.reshape(arr.shape[0], -1).T

    # -------------------------------------------------------------------------
    # Read mask
    # -------------------------------------------------------------------------

    with rasterio.open(mask_file) as msk:

        mask_arr = msk.read(1).ravel()
        mask_nodata = msk.nodatavals[0]

    if mask_arr.shape[0] != X_all.shape[0]:
        raise ValueError(
            f"Image and mask size mismatch for {img_base}: "
            f"image pixels = {X_all.shape[0]}, "
            f"mask {mask_arr.shape[0]}"
        )

    # -------------------------------------------------------------------------
    # Valid pixels
    # -------------------------------------------------------------------------

    if mask_nodata is not None:
        valid = mask_arr != mask_nodata
    else:
        valid = ~np.isnan(mask_arr)

    n_valid = int(valid.sum())

    prob_out = np.full(
        mask_arr.shape,
        out_nodata,
        dtype=np.float32
    )

    # -------------------------------------------------------------------------
    # Calculate probability
    # -------------------------------------------------------------------------

    if n_valid > 0:

        X_valid = X_all[valid]

        probabilities = loaded_rf.predict_proba(X_valid)

        # Same calculation as your original Code 2
        class_probability = np.max(
            probabilities,
            axis=1
        )

        prob_out[valid] = class_probability.astype(np.float32)

    # -------------------------------------------------------------------------
    # Reshape back to raster
    # -------------------------------------------------------------------------

    class_prob = prob_out.reshape(rows, cols)

    # -------------------------------------------------------------------------
    # Save output
    # -------------------------------------------------------------------------

    imgname = os.path.splitext(img_base)[0]

    out_path = os.path.join(
        dir_prob,
        f"{imgname}_{modelname}_prob.tif"
    )

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(class_prob, 1)

    return f"DONE: {out_path} | valid pixels: {n_valid:,}"

# =============================================================================
# Run probability calculation in parallel
# =============================================================================

results = []

with ThreadPoolExecutor(max_workers=tile_workers) as executor:

    futures = {
        executor.submit(probability_tile, img_file): img_file
        for img_file in img_files
    }

    for i, future in enumerate(as_completed(futures), start=1):

        try:
            message = future.result()
            results.append(message)
            print(f"{i}/{len(img_files)} - {message}")

        except Exception as e:

            img_file = futures[future]

            print(
                f"{i}/{len(img_files)} - FAILED: "
                f"{os.path.basename(img_file)}"
            )

            print(e)

print("\nProbability calculation completed.")

## Mosaic image tiles after processing

In [ ]:
dirpath = r"..."
out_tif = r"...\mosaic.tif"

vrt_file = out_tif.replace(".tif", ".vrt")

# Get input rasters
rasters = [
    os.path.join(dirpath, f)
    for f in os.listdir(dirpath)
    if f.lower().endswith(".tif")
]

print(f"Images to mosaic = {len(rasters)}")

# Build virtual mosaic
gdal.BuildVRT(vrt_file, rasters)

# Convert to GeoTIFF
gdal.Translate(
    out_tif,
    vrt_file,
    creationOptions=[
        "COMPRESS=LZW",
        "TILED=YES",
        "BIGTIFF=IF_SAFER"
    ]
)

print("Done")